# **LLM RAG Application**
This notebook uses the Langchain and OpenAI API to create RAG application. It read data from any URL, read all the link recursively and create vector store. Vectore is then used to asnwer the related queries. In current example, vectore store of Cylc8 documentation is created. 

In [ ]:
# Load necessary packages
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma, FAISS
import os
from utils import load_website, clean_text, chunk_text
from vectorstore import create_chroma_vectorstore, create_faiss_vectorstore
from retrievers import create_chroma_retrieval, create_faiss_retrieval

In [ ]:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

In [ ]:
# Add api-key
os.environ["OPENAI_API_KEY"] = "add-your-api-key"

# Define embedding model
embedding_model="text-embedding-3-large"

# Path where vectore store is supposed to store
vectorstore_parent_dir="add-storage-path"

In [ ]:
# Cylc8 documentation link
url = "https://cylc.github.io/cylc-doc/stable/html/index.html"

### **Load and pre-process the data and create chunks**

In [ ]:
# TODO: Should read all pages of website and create embeddings. For this we need more computational resources (Azure?), async routine or parallelisation?
data = load_website(url, max_pages=10)
print(f"\nLoaded {len(data)} documents total.")
#content = data[0].
# to load whole data
content = " ".join([doc.page_content for doc in data])

In [ ]:
# Clean text with unwanted tags
cleaned = clean_text(content)

In [ ]:
# Create chunks
chunks = chunk_text(
    cleaned, 
    chunk_size=1200, 
    overlap=200
    )

### **Create vectore store.** 

Create two different vector stores to access the performance: chromaDB and FAISS

**TODO: pinecone vestore store**

Idea is to test different vectorstores to access which one offers most efficient retrieval (or query response time). 

**TODO: To create embedding of large data**

To create embedding of large dataset, different methods can be used:
- batch processing
- Parallelization (Thread or Async)
- Cache
- Use GPU and local embedding model
- Use text-embedding-3-small

In [ ]:
# create and save embeddings to chromaDB
create_chroma_vectorstore(
    chunks, 
    vectorstore_parent_dir,
    embedding_model
    )

# create and save embeddings to FAISS
create_faiss_vectorstore(
    chunks,
    vectorstore_parent_dir,
    embedding_model
    )

#### **Now create retrievers and invoke with query**

Once the vectorstore is created, above code does not need to run again. Now, we need to create retriever and test the query

In [ ]:
model = OpenAIEmbeddings(model=embedding_model)

In [ ]:
# Write any query related to data
query = "Write any basic example of cylc8 workflow?"

In [ ]:
%%time
# Create Chromadb retrever and run a query
chroma_vectorstore = "chromadb"
persist_directory = os.path.join(vectorstore_parent_dir, chroma_vectorstore)
chroma_db = Chroma(
    persist_directory=persist_directory,
    embedding_function=model
)

qa_chain = create_chroma_retrieval(
    chroma_db, 
    model_name="gpt-4o", 
    temperature=0.2
    )
response = qa_chain.invoke({"query": query})

print("\nQuestion:", query)
print("\nAnswer:", response["result"])

In [ ]:
%%time
# Create faissdb retrever and run a query
faiss_vectorstore = "faissdb"
persist_directory = os.path.join(vectorstore_parent_dir, faiss_vectorstore)

faiss_db = FAISS.load_local(
    persist_directory, 
    model, 
    allow_dangerous_deserialization=True
    )

qa_chain = create_faiss_retrieval(
    faiss_db, 
    model_name="gpt-4o", 
    temperature=0.2
    )
response = qa_chain.invoke({"query": query})

print("\nQuestion:", query)
print("\nAnswer:", response["result"])
